In [8]:
# --- Бібліотеки для даної роботи ---
try:
    import pandas, jinja2
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install pandas jinja2

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Задача 2. Конфігуратор доступу до системи (RBAC)

**Умова:**

Ви розробляєте модуль авторизації (RBAC — Role-Based Access Control) для корпоративного порталу.

Доступ до різних секцій сайту визначається комбінацією п'яти параметрів користувача. Вам потрібно змоделювати цю логіку, перевірити всі можливі сценарії та відповісти на аналітичні питання.

**Вхідні параметри користувача (булеві змінні):**
1. `is_employee` — чи є співробітником компанії
2. `is_verified` — чи пройшов верифікацію пошти
3. `is_premium` — чи має преміум-підписку
4. `is_admin` — чи є адміністратором
5. `is_banned` — чи заблокований акаунт

**Правила доступу до секцій:**

Система має 4 рівні доступу. Доступ надається (`True`), якщо виконується логічна умова:
* **Base (Базовий):** Користувач є співробітником **ТА** верифікований **ТА НЕ** заблокований.
* **Premium (Преміум):** (Користувач є співробітником **АБО** має преміум) **ТА** верифікований **ТА НЕ** заблокований.
* **Admin (Адмін-панель):** Користувач є адміністратором **ТА** верифікований **ТА НЕ** заблокований.
* **Secret (Секретні матеріали):** (Користувач є адміністратором **АБО** (є співробітником **ТА** має преміум)) **ТА** верифікований **ТА НЕ** заблокований.

**Напишіть програму на Python, яка виконує наступні кроки:**
1.  Реалізуйте функцію

    ```python
    def check_access(is_employee, is_verified, is_premium, is_admin, is_banned):
        # Ваш код тут
        pass
    ```
    Функція повинна повертати словник вигляду `{'Base': ..., 'Premium': ..., 'Admin': ..., 'Secret': ...}`, де значення — це `True` або `False`.

2.  Згенеруйте **Таблицю істинності** для всіх 32 можливих комбінацій параметрів ($2^5 = 32$).
   *Підказка:* Використовуйте `itertools.product([True, False], repeat=5)` для генерації вхідних даних.

3. Виведіть таблицю у читабельному форматі (заголовок стовпчиків та рядки значень 0/1). Перші 5 стовпчиків це вхідні параметри користувача, а останні 4 це чи є доступ до відповідного рівня.

    ```
    Emp   Ver   Prem  Adm   Ban   | Base  Prem  Adm   Secr 
    -----------------------------------------------------------------
    1     1     1     1     1     | 0     0     0     0    
    1     1     1     1     0     | 1     1     1     1    
    1     1     1     0     1     | 0     0     0     0    
    # і далі всього 32 рядка
    ```

4.  Проаналізуйте результати та дайте відповіді на питання (в програмі чи текстом):
    * У скількох випадках користувач має повний доступ (до всіх 4 секцій одночасно)?
    * Чи існує комбінація, де користувач має доступ до **Premium**, але **не має** доступу до **Base**? Якщо так, виведіть параметри цієї комбінації та поясніть, чому так сталося.

In [9]:
import itertools
import pandas as pd
from IPython.display import display

def check_access(is_employee, is_verified, is_premium, is_admin, is_banned):
    is_safe = is_verified and not is_banned

    return {
        'Base': int(is_employee and is_safe),
        'Premium': int((is_employee or is_premium) and is_safe),
        'Admin': int(is_admin and is_safe),
        'Secret': int((is_admin or (is_employee and is_premium)) and is_safe)
    }

inputs = list(itertools.product([True, False], repeat=5))
data_rows = []
results = []
full_access_count = 0
prem_no_base_cases = []

for i, inp in enumerate(inputs, start=1):
    access = check_access(*inp)

    results.append((inp, access))

    row = {
        '№': i,
        'Emp': int(inp[0]),
        'Ver': int(inp[1]),
        'Prem': int(inp[2]),
        'Adm': int(inp[3]),
        'Ban': int(inp[4]),
        '|': '|', 
        'Base': access['Base'],
        'Premium': access['Premium'],
        'Admin': access['Admin'],
        'Secret': access['Secret']
    }
    data_rows.append(row)

    if all(access.values()):
        full_access_count += 1
    if access['Premium'] and not access['Base']:
        prem_no_base_cases.append(inp)

print("Красивий Вивід (Pandas):")
df = pd.DataFrame(data_rows)

def color_access(val):
    if val == 1:
        return 'background-color: #d4edda; color: #155724; font-weight: bold' # Зелений
    elif val == 0:
        return 'color: #lightgrey' # Сірий
    return ''

styled_table = df.style.map(color_access, subset=['Base', 'Premium', 'Admin', 'Secret']).hide()
display(styled_table) 

print("\n" + "="*60)
print(f"АНАЛІЗ РЕЗУЛЬТАТІВ:")
print(f"1. Кількість випадків повного доступу (всі 4 секції): {full_access_count}")
print(f"2. Випадки Premium=True, але Base=False (Знайдено: {len(prem_no_base_cases)}):")
for case in prem_no_base_cases:
    # case = (Emp, Ver, Prem, Adm, Ban)
    print(f"   -> Параметри: Emp={int(case[0])}, Ver={int(case[1])}, Prem={int(case[2])}, Adm={int(case[3])}, Ban={int(case[4])} (Клієнт з підпискою, не працівник)")
print("="*60 + "\n")

print("Технічний вивід (Текстова таблиця):")
header = f"{'Emp':^5} | {'Ver':^5} | {'Prem':^5} | {'Adm':^5} | {'Ban':^5} || {'Base':^5} | {'Prem':^5} | {'Adm':^5} | {'Secr':^5}"
print("-" * len(header))
print(header)
print("-" * len(header))

for inp, acc in results:
    input_str = f"{int(inp[0]):^5} | {int(inp[1]):^5} | {int(inp[2]):^5} | {int(inp[3]):^5} | {int(inp[4]):^5}"
    output_str = f"{int(acc['Base']):^5} | {int(acc['Premium']):^5} | {int(acc['Admin']):^5} | {int(acc['Secret']):^5}"
    print(f"{input_str} || {output_str}")

print("\nТехнічний вивід (Raw Data / List of Tuples):")
print(results)

Красивий Вивід (Pandas):


№,Emp,Ver,Prem,Adm,Ban,|,Base,Premium,Admin,Secret
1,1,1,1,1,1,|,0,0,0,0
2,1,1,1,1,0,|,1,1,1,1
3,1,1,1,0,1,|,0,0,0,0
4,1,1,1,0,0,|,1,1,0,1
5,1,1,0,1,1,|,0,0,0,0
6,1,1,0,1,0,|,1,1,1,1
7,1,1,0,0,1,|,0,0,0,0
8,1,1,0,0,0,|,1,1,0,0
9,1,0,1,1,1,|,0,0,0,0
10,1,0,1,1,0,|,0,0,0,0



АНАЛІЗ РЕЗУЛЬТАТІВ:
1. Кількість випадків повного доступу (всі 4 секції): 2
2. Випадки Premium=True, але Base=False (Знайдено: 2):
   -> Параметри: Emp=0, Ver=1, Prem=1, Adm=1, Ban=0 (Клієнт з підпискою, не працівник)
   -> Параметри: Emp=0, Ver=1, Prem=1, Adm=0, Ban=0 (Клієнт з підпискою, не працівник)

Технічний вивід (Текстова таблиця):
----------------------------------------------------------------------
 Emp  |  Ver  | Prem  |  Adm  |  Ban  || Base  | Prem  |  Adm  | Secr 
----------------------------------------------------------------------
  1   |   1   |   1   |   1   |   1   ||   0   |   0   |   0   |   0  
  1   |   1   |   1   |   1   |   0   ||   1   |   1   |   1   |   1  
  1   |   1   |   1   |   0   |   1   ||   0   |   0   |   0   |   0  
  1   |   1   |   1   |   0   |   0   ||   1   |   1   |   0   |   1  
  1   |   1   |   0   |   1   |   1   ||   0   |   0   |   0   |   0  
  1   |   1   |   0   |   1   |   0   ||   1   |   1   |   1   |   1  
  1   |   1   |   

### **Аналіз логіки та відповіді на питання**

Позначення змінних: $E$ (Employee), $V$ (Verified), $P$ (Premium), $A$ (Admin), $B$ (Banned).

Загальна умова безпеки (для всіх доступів): $S = V \land \neg B$ (Верифікований та Не заблокований).

#### Логічні вирази
1.  **Base:** $E \land S$
2.  **Premium:** $(E \lor P) \land S$
3.  **Admin:** $A \land S$
4.  **Secret:** $(A \lor (E \land P)) \land S$

#### Відповіді на питання (Аналітика)

**Питання 1: У скількох випадках користувач має повний доступ?**

Щоб мати всі 4 доступи, повинні виконуватися всі умови одночасно.
* $Base=1 \Rightarrow E=1$ (Користувач має бути співробітником).
* $Admin=1 \Rightarrow A=1$ (Користувач має бути адміном).
* $Secret=1 \Rightarrow$ Якщо $A=1$, то Secret автоматично 1.
* $Premium=1 \Rightarrow$ Якщо $E=1$, то Premium автоматично 1.
* $S=1 \Rightarrow V=1, B=0$.

Отже, параметри мають бути: $E=1, V=1, A=1, B=0$. Змінна $P$ (Premium-підписка) може бути будь-якою (0 або 1).

Це дає **2 випадки**:
1.  `1, 1, 1, 1, 0` (Співробітник-Адмін з підпискою)
2.  `1, 1, 0, 1, 0` (Співробітник-Адмін без підписки)

**Питання 2: Чи існує комбінація Premium=True, але Base=False?**

Перевіримо логічно:
* Для $Base=0$ потрібно, щоб $E=0$ (не співробітник) АБО ($V=0$ чи $B=1$).
* Але для $Premium=1$ обов'язково потрібні $V=1$ та $B=0$.
* Отже, єдиний варіант, щоб Base був 0, а Premium був 1 — це **$E=0$** (не співробітник).
* Якщо $E=0$, то формула Premium $(0 \lor P) \land 1 \land 1$ перетворюється на перевірку $P$. Тобто $P$ має бути 1.


Так, такі комбінації існують. Це звичайні клієнти, які купили підписку, але не працюють в компанії.

**Параметри:** $E=False, V=True, P=True, B=False$. ($A$ може бути будь-яким).

Це випадки, коли звичайний користувач купив сервіс.